# COMP4330 SSD300 fusion runner

Run cells from top to bottom. Formal runs must use a T4 GPU when available and write checkpoints to Google Drive. Do not evaluate `m1_fusion` or `control` on VOC2007 test until the configuration is frozen.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/itsvvy/comp4330-ssd-fusion.git"
PROJECT_DIR = Path("/content/comp4330-ssd-fusion")
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print("project:", PROJECT_DIR)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import platform
import shutil
import torch

print("python:", platform.python_version())
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda, torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram_gib:", torch.cuda.get_device_properties(0).total_memory / 2**30)
!nvidia-smi
print("disk free GiB:", shutil.disk_usage("/content").free / 2**30)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/COMP4330")
ASSET_ROOT = DRIVE_PROJECT / "assets"
DATA_ROOT = Path("/content/data/VOCdevkit")
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
print("assets:", ASSET_ROOT)
print("dataset:", DATA_ROOT)

In [ ]:
# First session only: archive the downloads on Drive and extract them to the
# local runtime disk. Later sessions skip existing archives.
!bash scripts/download_assets.sh "{ASSET_ROOT}" "{DATA_ROOT}"

In [ ]:
# Run a short profile before a formal training arm. Record GPU, iterations/s,
# and peak memory in the generated run directory.
!python scripts/run_experiment.py \
  --config configs/m1_fusion.yaml \
  --dataset-root "{DATA_ROOT}" \
  --save-root /content/drive/MyDrive/COMP4330/runs \
  --profile-iters 300

In [ ]:
# Change CONFIG to m0_ft.yaml, control.yaml, or m1_fusion.yaml.
CONFIG = "configs/m1_fusion.yaml"
RESUME = ""  # Optional absolute path to last.pt on Drive.

cmd = [
    "python", "scripts/run_experiment.py",
    "--config", CONFIG,
    "--dataset-root", str(DATA_ROOT),
    "--init-checkpoint", str(ASSET_ROOT / "weights/ssd300_mAP_77.43_v2.pth"),
    "--save-root", "/content/drive/MyDrive/COMP4330/runs",
]
if RESUME:
    cmd.extend(["--resume", RESUME])
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## After training

Keep every completed checkpoint and `metrics.jsonl`. After the configuration is frozen, run the evaluator once on VOC2007 test and store the detections plus the size-stratified metrics under the same run directory. Do not choose a result by repeatedly evaluating test and changing the training schedule.